# GlobalGates 채팅 / 게시글 보기 번역

저장된 채팅, 게시글, 댓글 본문을 사용자의 설정 언어로 번역해 보는 노트북이다. `member_language`, 원문, 기존 번역, Redis cache를 순서대로 확인한 뒤 LangChain 체인으로 번역한다.


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from redis import Redis

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. .env 파일 로드
load_dotenv()

# 2. 환경 변수에서 DB 정보 가져오기
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# 3. PostgreSQL 연결 URL 생성
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 4. 엔진 생성
engine = create_engine(db_url)


## 1. 실행 파라미터

실제 서비스에서는 `source_type`, `source_id`, `member_id`가 버튼 클릭 시 전달된다.

In [ ]:
SOURCE_TYPE = "post"   # chat or post
SOURCE_ID = 31
MEMBER_ID = 78

SOURCE_TYPE, SOURCE_ID, MEMBER_ID

## 2. member_language를 조회한다

번역 대상 언어는 글 작성자가 아니라, 지금 번역을 보는 회원의 설정 언어를 기준으로 잡는다.

In [ ]:
member_query = f"""
select member_language
from tbl_member
where id = {MEMBER_ID}
"""

member_df = pd.read_sql(member_query, engine)
member_df

In [ ]:
if member_df.empty:
    raise ValueError("member를 찾을 수 없습니다.")

member_language = str(member_df.loc[0, "member_language"]).strip()
target_language = member_language if member_language else "영어"

member_language, target_language


## 3. 원문을 단건 조회한다

채팅은 `tbl_message`, 게시글과 댓글은 `tbl_post`에서 읽는다.

In [ ]:
chat_query = f"""
select
    content,
    updated_datetime
from tbl_message
where id = {SOURCE_ID}
  and is_deleted = false
"""

post_query = f"""
select
    content,
    reply_post_id,
    updated_datetime
from tbl_post
where id = {SOURCE_ID}
  and post_status = 'active'
"""

In [ ]:
if SOURCE_TYPE == "post":
    source_df = pd.read_sql(post_query, engine)
else:
    source_df = pd.read_sql(chat_query, engine)

source_df.head()

In [ ]:
if source_df.empty:
    raise ValueError("원문을 찾을 수 없습니다.")

source_row = source_df.iloc[0].copy()

if SOURCE_TYPE == "chat":
    context_type = "chat"
    post_kind = ""
else:
    context_type = "post"
    post_kind = "post" if pd.isna(source_row["reply_post_id"]) else "reply"

context_type, post_kind, source_row["content"]


## 4. 기존 번역을 DB에서 단건 조회한다

번역 테이블이 있으면 기존 번역을 재사용하고, 없거나 원문 수정 시각이 다르면 새로 번역한다.

In [ ]:
if SOURCE_TYPE == "chat":
    translation_query = f"""
    select
        target_language,
        source_updated_datetime,
        translated_text
    from tbl_message_translation
    where message_id = {SOURCE_ID}
      and target_language = '{target_language}'
    """
else:
    translation_query = f"""
    select
        target_language,
        source_updated_datetime,
        translated_text
    from tbl_post_translation
    where post_id = {SOURCE_ID}
      and target_language = '{target_language}'
    """

try:
    translation_df = pd.read_sql(translation_query, engine)
except Exception:
    translation_df = pd.DataFrame(
        columns=[
            "target_language",
            "source_updated_datetime",
            "translated_text",
        ]
    )

translation_df.head()

## 5. Redis cache를 조회한다

같은 원문을 같은 언어로 다시 번역할 때는 Redis에 저장된 결과를 먼저 재사용한다.


In [ ]:
redis_client = Redis.from_url("redis://localhost:6380", decode_responses=True)

redis_client.ping()


In [ ]:
def make_translation_cache_key(source_type, source_id, target_language, updated_datetime):
    return f"translation:{source_type}:{source_id}:{target_language}:{updated_datetime}"


def translate_text(chain, target_language, context_type, post_kind, source_text):
    return chain.invoke(
        {
            "target_language": target_language,
            "context_type": context_type,
            "post_kind": post_kind,
            "source_text": source_text,
        }
    ).strip()


## 6. LangChain 체인을 만든다


In [ ]:
prompt_template = PromptTemplate.from_template(
    """
    너는 무역/거래 서비스 번역 비서다.

    아래 텍스트를 {target_language}로 자연스럽게 번역하라.
    의미를 바꾸지 말고, 숫자, 날짜, 수량, 국가명, 상품명, 문서명은 보존하라.

    규칙:
    - context_type이 chat이면 짧고 자연스러운 대화체
    - context_type이 post이고 post_kind가 post이면 정돈된 게시글 톤
    - context_type이 post이고 post_kind가 reply이면 짧고 자연스러운 댓글 톤

    source_text:
    {source_text}

    translated_text:
    """
)

llm = ChatOpenAI(
    model_name="gpt-5.4-nano",
    temperature=0
)

chain = prompt_template | llm | StrOutputParser()

In [ ]:
source_updated_datetime = pd.Timestamp(source_row["updated_datetime"]).isoformat()

redis_key = make_translation_cache_key(
    SOURCE_TYPE,
    SOURCE_ID,
    target_language,
    source_updated_datetime,
)

redis_text = redis_client.get(redis_key)

if redis_text:
    translated_text = redis_text
    translation_status = "redis_cached"
    print(f'캐시히트')
elif (
    not translation_df.empty
    and pd.Timestamp(translation_df.loc[0, "source_updated_datetime"]).isoformat() == source_updated_datetime
):
    translated_text = translation_df.loc[0, "translated_text"]
    translation_status = "db_cached"
    print(f'DB히트')
    redis_client.setex(
        redis_key,
        60 * 60 * 24,
        translated_text,
    )

else:
    translated_text = translate_text(
        chain,
        target_language,
        context_type,
        post_kind,
        source_row["content"],
    )
    translation_status = "generated"
    print(f'API호출')
    redis_client.setex(
        redis_key,
        60 * 60 * 24,
        translated_text,
    )

translated_text


## 7. 최종 결과를 확인한다

최종 결과는 DataFrame으로 한 번 더 정리해서 확인한다.


In [ ]:
result_row = {
    "source_type": SOURCE_TYPE,
    "source_id": SOURCE_ID,
    "member_id": MEMBER_ID,
    "member_language": member_language,
    "target_language": target_language,
    "translated_text": translated_text,
    "source_updated_datetime": source_updated_datetime,
    "translation_status": translation_status,
    "redis_key": redis_key,
}

translation_result_df = pd.DataFrame([result_row])
translation_result_df.head()


In [ ]:
print("source_type:", SOURCE_TYPE)
print("source_id:", SOURCE_ID)
print("member_id:", MEMBER_ID)
print("member_language:", member_language)
print("target_language:", target_language)
print("translation_status:", translation_status)
print("redis_key:", redis_key)
print("source_text:", source_row["content"])
print("translated_text:", translated_text)
